In [ ]:
import inspect
import fefetlab.b1500.driver as drv

print(inspect.signature(drv.B1500.dv))

In [ ]:
 # 单独测试通道4
from fefetlab.instruments.visa_session import VisaSession, VisaConfig
from fefetlab.b1500 import B1500

cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=60000,          # 先拉长，避免 10 s 太紧
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

with VisaSession(cfg) as s:
    b = B1500(s)

    print("IDN:", s.query("*IDN?"))
    print("ERRX start:", b.clear_err_queue())

    # 更接近手册的 high-speed spot 配置
    s.write("FMT 1")
    s.write("AV 10,1")
    s.write("FL 0")

    b.cn([4])

    # 你当前 driver 的 dv 签名是 dv(ch, vrange, voltage, compliance, ...)
    b.dv(4, 0, 0.0, 1e-3)

    raw_resp = s.query("TI 4,0")
    print("Raw TI response:", repr(raw_resp))

    print("ERRX end:", b.clear_err_queue())

    b.dz([4])
    b.cl([4])

In [ ]:
print(b._parse_scalar_response("NDI-010.895E-12"))

In [ ]:
import pandas as pd
import time

def spot_i_check_one_channel(b, ch, force_v=0.0, icomp=1e-3, delay_s=0.05):
    """
    对单个通道做最小 TI 点测检查。
    
    参数
    ----
    b : 你的 B1500 驱动对象
    ch : int
        通道号，例如 4 / 5 / 6 / 7
    force_v : float
        DV 输出电压，默认 0 V
    icomp : float
        电流 compliance，默认 1 mA
    delay_s : float
        每步之间稍微等一下，避免太快
    
    返回
    ----
    dict
        包含 ch, raw, current_A, quality, error
    """
    result = {
        "ch": ch,
        "raw": None,
        "current_A": None,
        "quality": None,
        "error": None,
    }

    try:
        # 1) 设置输出格式、平均、滤波
        b.write("FMT 1")
        time.sleep(delay_s)

        b.write("AV 10,1")
        time.sleep(delay_s)

        b.write("FL 0")
        time.sleep(delay_s)

        # 2) 开通道
        b.cn(ch)
        time.sleep(delay_s)

        # 3) 给这个通道加 0V
        b.dv(ch, 0, force_v, icomp)
        time.sleep(delay_s)

        # 4) 查询原始 TI 返回
        raw = b._query(f"TI {ch},0", check_err=False)
        result["raw"] = raw

        # 5) 解析数值
        val = b._parse_scalar_response(raw)
        result["current_A"] = val

        # 6) 简单质量标记
        if abs(val) <= 1e-6:
            result["quality"] = "valid"
        else:
            result["quality"] = "suspicious"

    except Exception as e:
        result["quality"] = "invalid"
        result["error"] = repr(e)

    finally:
        # 7) 收尾：尽量关输出/关通道
        try:
            b.dz(ch)
        except Exception:
            pass

        try:
            b.cl(ch)
        except Exception:
            pass

    return result

In [ ]:
channels = [4, 5, 6, 7]

rows = []

for ch in channels:
    print(f"\n========== 测试 CH{ch} ==========")
    res = spot_i_check_one_channel(b, ch, force_v=0.0, icomp=1e-3, delay_s=0.05)

    print("raw      =", repr(res["raw"]))
    print("current  =", res["current_A"])
    print("quality  =", res["quality"])
    print("error    =", res["error"])

    rows.append(res)

df = pd.DataFrame(rows)
df

In [ ]:
print(type(b))
print(dir(b))

In [ ]:
import time
import pandas as pd

def spot_i_check_one_channel(b, ch, force_v=0.0, icomp=1e-3, delay_s=0.05):
    result = {
        "ch": ch,
        "raw": None,
        "current_A": None,
        "quality": None,
        "error": None,
    }

    try:
        b._write("FMT 1")
        time.sleep(delay_s)

        b._write("AV 10,1")
        time.sleep(delay_s)

        b._write("FL 0")
        time.sleep(delay_s)

        # 这里改成 [ch]
        b.cn([ch])
        time.sleep(delay_s)

        b.dv(ch, 0, force_v, icomp)
        time.sleep(delay_s)

        raw = b._query(f"TI {ch},0", check_err=False)
        result["raw"] = raw

        val = b._parse_scalar_response(raw)
        result["current_A"] = val

        if abs(val) <= 1e-6:
            result["quality"] = "valid"
        else:
            result["quality"] = "suspicious"

    except Exception as e:
        result["quality"] = "invalid"
        result["error"] = repr(e)

    finally:
        try:
            b.dz([ch])
        except Exception:
            pass

        try:
            b.cl([ch])
        except Exception:
            pass

    return result

channels = [4, 5, 6, 7]
rows = []

for ch in channels:
    print(f"\n========== 测试 CH{ch} ==========")
    res = spot_i_check_one_channel(b, ch, force_v=0.0, icomp=1e-3, delay_s=0.05)

    print("raw      =", repr(res["raw"]))
    print("current  =", res["current_A"])
    print("quality  =", res["quality"])
    print("error    =", res["error"])

    rows.append(res)

df = pd.DataFrame(rows)

print("\n=== 汇总结果 ===")
print(df)

In [25]:
print("b =", b)
print("has s:", hasattr(b, "s"))
print("b.s =", b.s)
print("type(b.s) =", type(b.s))

try:
    print("IDN =", b.s.query("*IDN?"))
except Exception as e:
    print("query *IDN? 失败：", repr(e))

b = <fefetlab.b1500.driver.B1500 object at 0x000002119CD07460>
has s: True
b.s = <fefetlab.instruments.visa_session.VisaSession object at 0x000002119CD07020>
type(b.s) = <class 'fefetlab.instruments.visa_session.VisaSession'>
query *IDN? 失败： RuntimeError('VISA session is not opened.')


In [29]:
import pyvisa

VISA_ADDR = "GPIB0::17::INSTR"

rm = pyvisa.ResourceManager()
print("VISA resources:", rm.list_resources())

s = rm.open_resource(VISA_ADDR)

# 建议加上终止符和超时
s.timeout = 10000
s.write_termination = "\n"
s.read_termination = "\n"

print("*IDN? ->", s.query("*IDN?"))

# 把新的 session 绑回 b
b.s = s

print("重新绑定后，再试一次：")
print("*IDN? ->", b.s.query("*IDN?"))

VISA resources: ('GPIB0::17::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL7::INSTR', 'ASRL8::INSTR', 'ASRL9::INSTR', 'ASRL10::INSTR', 'ASRL11::INSTR')
*IDN? -> Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401
重新绑定后，再试一次：
*IDN? -> Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401
